# 오늘춥나?? — 3단계: 모델 5개 비교 (RF · LightGBM · LSTM · TSMixer)

4번 노트북의 랜덤포레스트는 **그 시각 만안 값 하나만** 봐요. 날씨는 몇 시간씩 흐름이 이어지니까, **직전 몇 시간을 같이 보면** 더 잘 맞힐 수 있는지 확인해요.
여기서는 하나를 고르지 않고 **다섯 개를 같은 조건으로 학습·채점해서 결과만 저장**해요. 고르는 건 결과를 보고 나서.

| 모델 | 보는 것 | 종류 |
|---|---|---|
| ① RF | 지금 시각 값 1개 (지금 사이트 모델) | 나무 평균 |
| ② RF + 지연특성 | 지금 + 직전 1~24시간 값·변화량·이동평균 | 나무 평균 |
| ③ LightGBM + 지연특성 | ②와 같은 입력 | 그래디언트 부스팅 (가볍고 빠름) |
| ④ LSTM | 직전 24시간 시퀀스 그대로 | 순환 신경망 |
| ⑤ TSMixer | 직전 24시간 시퀀스 그대로 | 2023년 Google 경량 시계열 모델 (MLP로 시간축·변수축을 번갈아 섞음) |

- **학습:** 2022~2023년 / **검증:** 2024년 / **최종 시험:** 2025년 · 모든 모델이 **같은 시험 줄**로 채점
- 현재 저장소 CSV에는 2022년이 없으므로, 2022년 행을 추가한 뒤 학습해야 해요.
- 모든 모델은 '안양대 − 만안' **보정량**을 배워요 (4번 노트북과 같음)

**런타임 → 런타임 유형 변경 → T4 GPU**로 바꾸면 ④⑤가 빨라져요 (CPU도 동작, 약 5분).

**순서:** ① 준비 → ② 시간 맞추기·나누기 → ③ 채점 기준 → ④ 입력 만들기 → ⑤ 나무 모델 3개 → ⑥ 신경망 2개 → ⑦ 비교 → ⑧ 저장

## ① 준비: 라이브러리 · 드라이브 · 데이터

In [ ]:
!pip -q install lightgbm

import time
import numpy as np
import pandas as pd
import torch
from torch import nn
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from google.colab import drive

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("신경망 장치:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(GPU 없음, CPU로 진행)")

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/onulchupna"
raw = pd.read_csv(f"{BASE}/train_data.csv", parse_dates=["tm"])
print(len(raw), "줄,", raw.tm.min(), "~", raw.tm.max())

## ② 1시간 간격 맞추기 · 학습/시험 나누기

시퀀스 모델은 '직전 24시간'을 보니까 시간이 1시간 간격으로 빈틈없이 이어져야 해요.
- 빠진 시각(3년 중 몇십 군데, 대부분 1~3시간)은 줄을 만들고, **만안 입력만** 앞뒤 값으로 이어 채워요 (9시간 이하).
- 안양대 정답은 채우지 않아요. **원래 있던 줄만** 학습·채점에 써요.
- 직전 24시간이 다 있는 줄만 써서, 다섯 모델이 정확히 같은 줄로 채점돼요.

In [ ]:
TRAIN_START_YEAR = 2022
VAL_YEAR = 2024
TEST_YEAR = 2025
WINDOW = 24          # ④⑤ 신경망이 보는 직전 시간 수
TARGETS = ["uni_ta", "uni_hm", "uni_ws"]
BASES = ["aws_ta", "aws_hm", "aws_ws"]       # 각 정답의 기준이 되는 만안 값
NOW = ["aws_ta", "aws_hm", "aws_ws", "hour_sin", "hour_cos", "month_sin", "month_cos"]

df = raw.set_index("tm").sort_index()
df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="h"))
df["observed"] = df["uni_ta"].notna()
df[BASES] = df[BASES].interpolate(limit=9, limit_area="inside")
df["month"], df["hour"] = df.index.month, df.index.hour
df = df.rename_axis("tm").reset_index()
df["hour_sin"], df["hour_cos"] = np.sin(2 * np.pi * df.hour / 24), np.cos(2 * np.pi * df.hour / 24)
df["month_sin"], df["month_cos"] = np.sin(2 * np.pi * df.month / 12), np.cos(2 * np.pi * df.month / 12)

need = max(WINDOW, 25)   # 지연특성이 24시간 전까지 보므로 최소 25시간
full_history = df[BASES].notna().all(axis=1).rolling(need).sum().eq(need)
usable = df.observed & full_history
year = df.tm.dt.year
tr_rows = np.where(usable & (year >= TRAIN_START_YEAR) & (year < VAL_YEAR))[0]
va_rows = np.where(usable & (year == VAL_YEAR))[0]
te_rows = np.where(usable & (year == TEST_YEAR))[0]
train, val, test = df.iloc[tr_rows], df.iloc[va_rows], df.iloc[te_rows]
D_tr = train[TARGETS].to_numpy(float) - train[BASES].to_numpy(float)    # 정답: 보정량
D_val = val[TARGETS].to_numpy(float) - val[BASES].to_numpy(float)
base_te = test[BASES].to_numpy(float)
print(f"학습 {len(train):,}시간 ({train.tm.min():%Y-%m} ~ {train.tm.max():%Y-%m})")
print(f"검증 {len(val):,}시간 ({VAL_YEAR}년)")
print(f"시험 {len(test):,}시간 ({test.tm.min():%Y-%m} ~ {test.tm.max():%Y-%m})")

## ③ 채점 기준 (4번 노트북과 같음)

예측한 기온·습도·풍속으로 기상청 체감온도를 계산해, 실제 안양대 체감온도와 비교해요.
- **±1°C 적중률**: 사람이 거의 구분 못 하는 차이 → **메인 점수**
- **겨울 ±1°C**: 10~4월. 풍속 1.3 m/s 기준선 때문에 가장 어려운 구간

In [ ]:
def kma_winter(ta, ws):
    v = np.asarray(ws) * 3.6
    wc = 13.12 + 0.6215 * ta - 11.37 * v**0.16 + 0.3965 * v**0.16 * ta
    return np.where((np.asarray(ta) <= 10) & (np.asarray(ws) >= 1.3), wc, ta)

def kma_summer(ta, rh):
    tw = (ta * np.arctan(0.151977 * np.sqrt(rh + 8.313659)) + np.arctan(ta + rh)
          - np.arctan(rh - 1.67633) + 0.00391838 * rh**1.5 * np.arctan(0.023101 * rh) - 4.686035)
    return -0.2442 + 0.55399 * tw + 0.45535 * ta - 0.0022 * tw**2 + 0.00278 * tw * ta + 3.0

def kma_feels(month, ta, rh, ws):
    month, ta, rh, ws = (np.asarray(x, dtype=float) for x in (month, ta, rh, ws))
    return np.where((month >= 5) & (month <= 9), kma_summer(ta, rh), kma_winter(ta, ws))

def tidy(pred):
    pred = np.array(pred, dtype=float)
    pred[:, 1] = pred[:, 1].clip(0, 100)   # 습도 0~100%
    pred[:, 2] = pred[:, 2].clip(0, None)  # 풍속 0 이상
    return pred

real_feels = kma_feels(test.month, test.uni_ta, test.uni_hm, test.uni_ws)
winter = ~test.month.between(5, 9).to_numpy()

def score(pred):
    err = np.abs(kma_feels(test.month, pred[:, 0], pred[:, 1], pred[:, 2]) - real_feels)
    hit1, hit2 = (err <= 1).mean() * 100, (err <= 2).mean() * 100
    winter_hit1, feel_mae = (err[winter] <= 1).mean() * 100, err.mean()
    mae_score = np.clip(1 - feel_mae / 3, 0, 1) * 100
    overall = .50 * hit1 + .20 * winter_hit1 + .15 * hit2 + .15 * mae_score
    return {
        "종합 점수": overall,
        "체감 ±1°C(%)": hit1,
        "체감 ±2°C(%)": hit2,
        "겨울 ±1°C(%)": winter_hit1,
        "체감 오차(°C)": feel_mae,
        "기온 오차(°C)": np.abs(pred[:, 0] - test.uni_ta).mean(),
        "습도 오차(%)": np.abs(pred[:, 1] - test.uni_hm).mean(),
        "풍속 오차(m/s)": np.abs(pred[:, 2] - test.uni_ws).mean(),
    }

preds = {"보정 안 함(날씨앱)": base_te}   # 모델별 2025년 최종 시험 예측 [기온, 습도, 풍속]
info = {}                                  # 모델별 학습 시간 등

## ④ 입력 만들기

**나무 모델용 지연특성 (②③):** 지금 값 + 1·2·3·6·12·24시간 전 값, 1·3시간 변화량, 3·6·24시간 평균, 6시간 흔들림(표준편차), 하루 기온 폭, 6시간 최대 풍속.

**신경망용 시퀀스 (④⑤):** 직전 24시간 × 7개 값(만안 3개 + 시간·월 sin·cos)을 그대로. 평균 0, 표준편차 1로 맞춰요 (학습 기간 기준).

In [ ]:
lag = df[NOW].copy()
for c in BASES:
    s = df[c]
    for k in (1, 2, 3, 6, 12, 24):
        lag[f"{c}_lag{k}"] = s.shift(k)
    lag[f"{c}_d1"], lag[f"{c}_d3"] = s - s.shift(1), s - s.shift(3)
    for w in (3, 6, 24):
        lag[f"{c}_mean{w}"] = s.rolling(w).mean()
    lag[f"{c}_std6"] = s.rolling(6).std()
lag["ta_range24"] = df.aws_ta.rolling(24).max() - df.aws_ta.rolling(24).min()
lag["ws_max6"] = df.aws_ws.rolling(6).max()
L_tr, L_val, L_te = lag.iloc[tr_rows], lag.iloc[va_rows], lag.iloc[te_rows]
print("지연특성", lag.shape[1], "개")

seq = df[NOW].to_numpy(np.float32)
mu, sd = seq[tr_rows].mean(0), seq[tr_rows].std(0) + 1e-6
seq = (seq - mu) / sd
def windows(rows):
    return seq[rows[:, None] + np.arange(-WINDOW + 1, 1)[None, :]]
X_tr, X_val, X_te = windows(tr_rows), windows(va_rows), windows(te_rows)
y_mu, y_sd = D_tr.mean(0), D_tr.std(0) + 1e-6
Y_tr = ((D_tr - y_mu) / y_sd).astype(np.float32)
Y_val = ((D_val - y_mu) / y_sd).astype(np.float32)
print("시퀀스", X_tr.shape, "(줄, 시간, 값)")

## ⑤ 나무 모델 3개

In [ ]:
RF_PARAMS = dict(n_estimators=100, min_samples_leaf=20, max_depth=14, n_jobs=-1, random_state=0)

def fit_trees(name, make, X_train, X_test):
    t0 = time.time()
    delta = np.column_stack([make().fit(X_train, D_tr[:, j]).predict(X_test) for j in range(3)])
    preds[name] = tidy(base_te + delta)
    info[name] = f"{time.time() - t0:.0f}초"
    print(f"{name:<22} 완료 ({info[name]})")

fit_trees("① RF (지금 사이트)", lambda: RandomForestRegressor(**RF_PARAMS), train[NOW], test[NOW])
fit_trees("② RF + 지연특성", lambda: RandomForestRegressor(**RF_PARAMS), L_tr, L_te)
fit_trees("③ LightGBM + 지연특성", lambda: lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=40,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
    objective="huber", alpha=1.0, verbose=-1, random_state=0), L_tr, L_te)

## ⑥ 신경망 2개: LSTM · TSMixer

- 두 모델 모두 마지막에 **지금 시각 입력을 한 번 더** 붙여줘요. 지금 값이 가장 중요한 단서라서.
- **2024년 전체를 검증 세트**로 사용해, 검증 점수가 8에폭 동안 안 좋아지면 멈춰요 (조기 종료). 2025년 시험값은 체크포인트 선택에 쓰지 않아요.
- 신경망은 시작값(시드)에 따라 결과가 조금씩 달라서, `SEEDS`개를 학습해 **평균**내요. 빨리 보려면 1, 발표용은 3~5.

In [ ]:
SEEDS = 3

class LSTMNet(nn.Module):
    def __init__(self, c, hidden=64, layers=2, drop=0.1):
        super().__init__()
        self.lstm = nn.LSTM(c, hidden, layers, batch_first=True, dropout=drop)
        self.head = nn.Sequential(nn.Linear(hidden + c, 64), nn.GELU(), nn.Linear(64, 3))
    def forward(self, x):
        h, _ = self.lstm(x)
        return self.head(torch.cat([h[:, -1], x[:, -1]], dim=1))

class MixerBlock(nn.Module):
    def __init__(self, L, c, ff, drop=0.1):
        super().__init__()
        self.norm_t, self.norm_f = nn.LayerNorm(c), nn.LayerNorm(c)
        self.time = nn.Sequential(nn.Linear(L, L), nn.GELU(), nn.Dropout(drop))
        self.feat = nn.Sequential(nn.Linear(c, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, c), nn.Dropout(drop))
    def forward(self, x):                                                    # x: (배치, 시간, 값)
        x = x + self.time(self.norm_t(x).transpose(1, 2)).transpose(1, 2)    # 시간축 섞기
        return x + self.feat(self.norm_f(x))                                 # 변수축 섞기

class TSMixer(nn.Module):
    def __init__(self, L, c, hidden=32, blocks=3, drop=0.1):
        super().__init__()
        self.proj = nn.Linear(c, hidden)
        self.blocks = nn.Sequential(*[MixerBlock(L, hidden, 2 * hidden, drop) for _ in range(blocks)])
        self.head = nn.Sequential(nn.Linear(L * hidden + c, 64), nn.GELU(), nn.Linear(64, 3))
    def forward(self, x):
        z = self.blocks(self.proj(x)).flatten(1)
        return self.head(torch.cat([z, x[:, -1]], dim=1))

def train_net(make, seed, epochs=60, patience=8, lr=2e-3, bs=256):
    torch.manual_seed(seed)
    Xt, Yt = torch.tensor(X_tr, device=DEVICE), torch.tensor(Y_tr, device=DEVICE)
    Xv, Yv = torch.tensor(X_val, device=DEVICE), torch.tensor(Y_val, device=DEVICE)
    net = make().to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    loss_fn = nn.HuberLoss()
    best, best_state, bad = np.inf, None, 0
    g = torch.Generator(device="cpu").manual_seed(seed)
    for ep in range(1, epochs + 1):
        net.train()
        for b in torch.randperm(len(Xt), generator=g).split(bs):
            b = b.to(DEVICE)
            opt.zero_grad()
            loss_fn(net(Xt[b]), Yt[b]).backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        sched.step()
        net.eval()
        with torch.no_grad():
            v = loss_fn(net(Xv), Yv).item()
        if v < best - 1e-4:
            best, bad, best_ep = v, 0, ep
            best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        out = torch.cat([net(torch.tensor(X_te[i:i + 4096], device=DEVICE)) for i in range(0, len(X_te), 4096)])
    print(f"    시드 {seed}: {best_ep}에폭이 최고 (검증 손실 {best:.4f})")
    return out.cpu().numpy(), net

def fit_net(name, make):
    t0 = time.time()
    print(f"{name} 학습 중 (시드 {SEEDS}개)")
    outs = [train_net(make, s) for s in range(SEEDS)]
    delta = np.mean([o for o, _ in outs], 0) * y_sd + y_mu
    preds[name] = tidy(base_te + delta)
    n_params = sum(p.numel() for p in outs[0][1].parameters())
    info[name] = f"{time.time() - t0:.0f}초 · 파라미터 {n_params / 1000:.0f}K × {SEEDS}"
    print(f"{name} 완료 ({info[name]})\n")

C = len(NOW)
fit_net("④ LSTM", lambda: LSTMNet(C))
fit_net("⑤ TSMixer", lambda: TSMixer(WINDOW, C))

## ⑦ 비교

초록이 진할수록 좋은 값이에요 (적중률은 높을수록, 오차는 낮을수록). `RF 대비`는 지금 사이트 모델보다 ±1°C 적중률이 몇 %p 나은지.

**종합 점수(0~100):** ±1°C 적중률 50% + 겨울 ±1°C 20% + ±2°C 15% + 체감 MAE 점수 15%. MAE 점수는 오차 0°C=100점, 3°C 이상=0점이에요. 고정식이라 다른 실행끼리도 비교할 수 있어요.

In [ ]:
REF = "① RF (지금 사이트)"
result = pd.DataFrame({name: score(p) for name, p in preds.items()}).T
result["RF 대비(%p)"] = result["체감 ±1°C(%)"] - result.loc[REF, "체감 ±1°C(%)"]
result["학습"] = pd.Series(info)
result = result.fillna({"학습": "-"})

higher = ["종합 점수", "체감 ±1°C(%)", "체감 ±2°C(%)", "겨울 ±1°C(%)", "RF 대비(%p)"]
lower = ["체감 오차(°C)", "기온 오차(°C)", "습도 오차(%)", "풍속 오차(m/s)"]
models_only = result.index != "보정 안 함(날씨앱)"
(result.style
    .format({**{c: "{:.1f}" for c in higher}, **{c: "{:.2f}" for c in lower}})
    .background_gradient("Greens", subset=pd.IndexSlice[models_only, higher])
    .background_gradient("Greens_r", subset=pd.IndexSlice[models_only, lower])
    .set_caption(f"{TEST_YEAR}년 {len(test):,}시간 시험"))

### 계절별 ±1°C 적중률

모델마다 강한 계절이 다를 수 있어요. 겨울은 풍속, 여름은 습도가 체감온도를 좌우해요.

In [ ]:
season = pd.Series(np.select([test.month.isin([12, 1, 2]), test.month.isin([3, 4, 5]), test.month.isin([6, 7, 8])],
                             ["겨울", "봄", "여름"], "가을"), index=test.index)
by_season = {}
for name, p in preds.items():
    hit = pd.Series(np.abs(kma_feels(test.month, p[:, 0], p[:, 1], p[:, 2]) - real_feels) <= 1, index=test.index)
    by_season[name] = hit.groupby(season).mean() * 100
by_season = pd.DataFrame(by_season).T[["봄", "여름", "가을", "겨울"]]
by_season.style.format("{:.1f}").background_gradient("Greens", subset=pd.IndexSlice[by_season.index != "보정 안 함(날씨앱)", :])

### 하루 따라가 보기

2025년 시험 기간의 한 날을 골라 모델들이 시간별 캠퍼스 체감온도를 어떻게 예측했는지 그려봐요. `DAY`를 바꿔보세요.
(그래프 글씨는 코랩에 한글 글꼴이 없어 영어로 써요.)

In [ ]:
import matplotlib.pyplot as plt

DAY = "2025-01-09"
SHORT = {"보정 안 함(날씨앱)": "Weather app", "① RF (지금 사이트)": "RF", "② RF + 지연특성": "RF+lag",
         "③ LightGBM + 지연특성": "LightGBM+lag", "④ LSTM": "LSTM", "⑤ TSMixer": "TSMixer"}
m = (test.tm.dt.strftime("%Y-%m-%d") == DAY).to_numpy()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test.hour[m], real_feels[m], "k-", lw=2.5, label="Campus (actual)")
for name, p in preds.items():
    f = kma_feels(test.month, p[:, 0], p[:, 1], p[:, 2])
    ax.plot(test.hour[m], f[m], "--" if name.startswith("보정") else "-", lw=1.3, label=SHORT.get(name, name))
ax.set(xlabel="hour", ylabel="feels-like (°C)", title=f"{DAY} campus feels-like temperature", xticks=range(0, 24, 2))
ax.legend(ncol=4, fontsize=9)
ax.grid(alpha=0.3)
plt.show()

## ⑧ 저장 (모델은 고르지 않음)

비교 결과와 **모델별 2025년 최종 시험 예측값**을 드라이브에 저장해요. 나중에 사이트에 붙일 모델을 고를 때 이 파일들을 보고 정해요.
- `compare_result.csv`: ⑦의 표
- `compare_season.csv`: 계절별 표
- `compare_predictions.csv`: 2025년 시간별 실제값 + 모델별 예측 (기온·습도·풍속·체감)

`campus_model.joblib`(지금 사이트 모델)은 **건드리지 않아요.**

In [ ]:
result.to_csv(f"{BASE}/compare_result.csv", encoding="utf-8-sig")
by_season.to_csv(f"{BASE}/compare_season.csv", encoding="utf-8-sig")

out = test[["tm", *BASES, *TARGETS]].reset_index(drop=True)
out["real_feels"] = real_feels
for name, p in preds.items():
    key = SHORT.get(name, name)
    out[[f"{key}_ta", f"{key}_hm", f"{key}_ws"]] = p
    out[f"{key}_feels"] = kma_feels(test.month, p[:, 0], p[:, 1], p[:, 2])
out.to_csv(f"{BASE}/compare_predictions.csv", index=False, encoding="utf-8-sig")
print("저장:", f"{BASE}/compare_result.csv, compare_season.csv, compare_predictions.csv")